# 04 — Entrenamiento con cinco etiquetas de daño o SEGURO

Este cuaderno compara modelos locales de *machine learning* para un problema multietiqueta grueso. Las salidas operativas son cinco tipos de daño —racismo/discriminación, acoso por género o identidad, acoso personal, amenaza directa y contenido sexual— o `SEGURO` cuando no se predice ningún daño.

Las 14 etiquetas finas **no se entrenan como salidas** ni se incorporan al texto. Solo cumplen dos funciones legítimas: (1) construir determinísticamente las etiquetas gruesas y (2) auditar que fenómenos raros estén representados en las particiones. Los flags `ironia_ambigua`, `humor_encubridor` y `contexto_necesario` tampoco son clases temáticas: se usan para calibrar `needs_review`.

## 0. Dependencias

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.13,<2" "scikit-learn>=1.5,<2" "joblib>=1.4,<2" "matplotlib>=3.9,<4"

## 1. Configuración reproducible

In [ ]:
from pathlib import Path
import gc
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'scripts_auxiliares' / 'modelos_gruesos_moderador.py').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('No se encontró la raíz del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts_auxiliares.flujo_hibrido_moderador import (
    build_hybrid_dataset, load_human_holdout, load_taxonomy,
)
from scripts_auxiliares.modelos_gruesos_moderador import (
    COARSE_ORDER, DAMAGE_ORDER, FINE_TO_COARSE, MODEL_LABELS, MODEL_ORDER,
    add_coarse_targets, balanced_group_split_search, evaluate_candidate,
    fit_candidate, load_coarse_model, review_routing_curve,
    save_coarse_model, select_review_margin, split_prevalence_table,
    target_matrix, tune_candidate,
)

SEED = 42
SPLIT_SEARCH_TRIALS = 250
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
MAX_FEATURES = 50_000
FLASH_PSEUDO_WEIGHT = 0.50
FLASH_REVIEW_THRESHOLD = 0.90
MIN_DAMAGE_MACRO_F1 = 0.70
MIN_DAMAGE_RECALL = 0.80
MIN_FLAG_CAPTURE = 0.80

MODEL_DIR = ROOT / 'modelos' / 'moderador_grueso'
METRICS_DIR = ROOT / 'resultados' / 'metricas' / 'moderador_grueso'
FIGURES_DIR = ROOT / 'resultados' / 'figuras' / 'moderador_grueso'
for directory in (MODEL_DIR, METRICS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print('Raíz:', ROOT)

## 2. Etiqueta híbrida humano grueso > Pro > Flash y exclusión del holdout

Pro reemplaza a Flash en cada chunk revisado. En los 139 casos de duda persistente de Pro, la adjudicación humana reemplaza solo el objetivo grueso y conserva las etiquetas finas de Pro exclusivamente como referencia auditable. Las categorías transversales continúan separadas de las seis categorías base. La construcción se bloquea hasta completar los 139 casos para impedir un reentrenamiento parcial. Un eventual consenso humano externo sigue siendo holdout y nunca participa en selección de modelo ni ajuste de umbrales.

In [ ]:
taxonomy, FINE_ORDER, FLAG_ORDER = load_taxonomy(ROOT)
hybrid_df, hybrid_meta, _, _ = build_hybrid_dataset(
    ROOT,
    recalibrated_threshold=FLASH_REVIEW_THRESHOLD,
    flash_weight=FLASH_PSEUDO_WEIGHT,
    write_output=True,
)
hybrid_df = add_coarse_targets(hybrid_df, taxonomy)
human_ids = set(hybrid_df.loc[hybrid_df['human_holdout'], 'chunk_id'])
modeling_pool = hybrid_df.loc[~hybrid_df['human_holdout']].reset_index(drop=True)
assert human_ids.isdisjoint(set(modeling_pool['chunk_id']))
assert len(COARSE_ORDER) == 6 and len(DAMAGE_ORDER) == 5

print(f'Corpus: {len(hybrid_df):,}; modelado: {len(modeling_pool):,}; holdout humano: {len(human_ids):,}.')
display(modeling_pool['label_source'].value_counts().rename_axis('fuente').to_frame('chunks'))

## 3. Mapeo reproducible: etiquetas finas → cinco daños o seguro

El mapeo conserva toda coocurrencia: un chunk puede ser, por ejemplo, racista y acosador. `SEGURO` es excluyente y se asigna únicamente cuando no existe daño. Las etiquetas finas no son predictores; usar la respuesta anotada como feature produciría fuga de información.

In [ ]:
mapping_table = pd.DataFrame(
    [{'etiqueta_fina': fine, 'objetivo_grueso': coarse} for fine, coarse in FINE_TO_COARSE.items()]
).sort_values(['objetivo_grueso', 'etiqueta_fina'])
display(mapping_table)
mapping_table.to_csv(METRICS_DIR / 'mapeo_fino_a_grueso.csv', index=False)

coarse_rows = []
for category in COARSE_ORDER:
    mask = modeling_pool['coarse_labels'].map(lambda values: category in values)
    coarse_rows.append({
        'categoria': category,
        'positivos': int(mask.sum()),
        'prevalencia': float(mask.mean()),
        'videos': int(modeling_pool.loc[mask, 'video_id'].nunique()),
        'proporcion_pro': float((modeling_pool.loc[mask, 'label_source'] == 'pro').mean()),
        'peso_efectivo': float(modeling_pool.loc[mask, 'sample_weight'].sum()),
    })
coarse_distribution = pd.DataFrame(coarse_rows)
display(coarse_distribution)
coarse_distribution.to_csv(METRICS_DIR / 'distribucion_objetivos_gruesos.csv', index=False)

## 4. Partición agrupada y balanceada

Una división aleatoria por chunk filtraría expresiones del mismo video. Se generan 250 candidatos con `GroupShuffleSplit`, manteniendo cada video en una sola partición, y se elige el que minimiza diferencias de prevalencia. La auditoría considera las seis salidas gruesas, las etiquetas finas y los flags para evitar celdas vacías; solamente las seis gruesas se entrenan. La proporción objetivo es 70%/15%/15%.

In [ ]:
split_indices, split_meta = balanced_group_split_search(
    modeling_pool, FINE_ORDER, FLAG_ORDER, seed=SEED,
    test_size=TEST_SIZE, validation_size=VALIDATION_SIZE,
    trials=SPLIT_SEARCH_TRIALS,
)
train_df = modeling_pool.iloc[split_indices['train']].reset_index(drop=True)
validation_df = modeling_pool.iloc[split_indices['validation']].reset_index(drop=True)
test_df = modeling_pool.iloc[split_indices['test']].reset_index(drop=True)
split_frames = {'train': train_df, 'validation': validation_df, 'test': test_df}
video_sets = {name: set(frame['video_id']) for name, frame in split_frames.items()}
assert video_sets['train'].isdisjoint(video_sets['validation'])
assert video_sets['train'].isdisjoint(video_sets['test'])
assert video_sets['validation'].isdisjoint(video_sets['test'])
assert human_ids.isdisjoint(set(pd.concat(split_frames.values())['chunk_id']))

split_prevalence = split_prevalence_table(split_frames)
display(pd.Series(split_meta).to_frame('valor'))
display(split_prevalence)
split_prevalence.to_csv(METRICS_DIR / 'prevalencia_por_particion.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(COARSE_ORDER))
width = 0.25
for offset, name in enumerate(['train', 'validation', 'test']):
    values = (
        split_prevalence.loc[split_prevalence['particion'] == name]
        .set_index('categoria').loc[COARSE_ORDER, 'prevalencia'].to_numpy()
    )
    ax.bar(x + (offset - 1) * width, values, width, label=name)
ax.set_yscale('log')
ax.set_ylabel('Prevalencia (escala logarítmica)')
ax.set_title('Balance de las cinco etiquetas de daño y SEGURO por partición')
ax.set_xticks(x, [name.replace('_', '\n') for name in COARSE_ORDER], rotation=0)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'balance_particiones_gruesas.png', dpi=170, bbox_inches='tight')
plt.show()

![Balance de las particiones](../resultados/figuras/moderador_grueso/balance_particiones_gruesas.png)

## 5. Modelos comparados y regla de selección

Todos reciben exactamente los mismos textos, particiones y pesos por fuente. Se comparan cinco niveles:

1. `Dummy (prior)`: ignora el texto; demuestra por qué accuracy es engañosa.
2. `Complement NB`: baseline probabilístico eficiente para texto desbalanceado.
3. `Regresión logística`: modelo lineal discriminativo con TF‑IDF de palabras.
4. `SVM lineal palabra+carácter`: combina palabras y fragmentos ortográficos, útil ante modismos y variantes.
5. `Gradient boosting + SVD`: modelo no lineal sobre una representación semántica reducida.

Los umbrales se ajustan únicamente en validación, maximizando F1 por salida. El ganador se selecciona por **F1 macro de las cinco etiquetas de daño** y PR‑AUC macro de daño como desempate; `SEGURO` queda fuera del criterio primario. La prueba no interviene en la elección. El balanceo de clases se aplica dentro del entrenamiento; validación y prueba conservan la distribución observada.

In [ ]:
candidate_models = {}
validation_reports = {}
validation_scores = {}
benchmark_rows = []

for model_name in MODEL_ORDER:
    print('Entrenando:', MODEL_LABELS[model_name])
    model, diagnostics = fit_candidate(model_name, train_df, max_features=MAX_FEATURES)
    tuned_scores = tune_candidate(model, validation_df)
    metrics, report, scores = evaluate_candidate(model, validation_df)
    candidate_models[model_name] = model
    validation_reports[model_name] = report
    validation_scores[model_name] = scores
    benchmark_rows.append({
        **diagnostics,
        **{f'val_{key}': value for key, value in metrics.items()},
    })

validation_benchmark = pd.DataFrame(benchmark_rows).sort_values(
    ['val_damage_f1_macro', 'val_damage_pr_auc_macro'], ascending=False
).reset_index(drop=True)
WINNER_NAME = validation_benchmark.iloc[0]['model']
winner_development = candidate_models[WINNER_NAME]
display(validation_benchmark[[
    'model_label', 'features', 'training_seconds', 'val_damage_f1_macro',
    'val_damage_f1_micro', 'val_damage_pr_auc_macro', 'val_exact_match',
]])
print('Ganador determinado solo con validación:', MODEL_LABELS[WINNER_NAME])
validation_benchmark.to_csv(METRICS_DIR / 'comparacion_validacion.csv', index=False)

## 6. Comparación final en prueba por video

La prueba se abre después de seleccionar el ganador. Se muestran todos los candidatos para describir el experimento, pero la decisión permanece fijada por validación. Se priorizan F1 macro, PR‑AUC y sensibilidad de cada daño; `exact_match` se informa, pero no decide porque `SEGURO` domina el corpus.

In [ ]:
test_rows = []
test_reports = {}
test_scores = {}
for model_name in MODEL_ORDER:
    metrics, report, scores = evaluate_candidate(candidate_models[model_name], test_df)
    test_reports[model_name] = report
    test_scores[model_name] = scores
    test_rows.append({
        'model': model_name, 'model_label': MODEL_LABELS[model_name],
        **{f'test_{key}': value for key, value in metrics.items()},
    })
test_benchmark = pd.DataFrame(test_rows)
benchmark = validation_benchmark.merge(test_benchmark, on=['model', 'model_label'])
benchmark = benchmark.sort_values(['val_damage_f1_macro', 'val_damage_pr_auc_macro'], ascending=False)
display(benchmark[[
    'model_label', 'val_damage_f1_macro', 'test_damage_f1_macro',
    'test_damage_recall_micro', 'test_damage_pr_auc_macro',
    'test_exact_match', 'test_milliseconds_per_1000',
]])
benchmark.to_csv(METRICS_DIR / 'comparacion_modelos_completa.csv', index=False)
for name, report in test_reports.items():
    report.to_csv(METRICS_DIR / f'reporte_prueba_{name}.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_order = benchmark['model'].tolist()
labels = [MODEL_LABELS[name] for name in plot_order]
x = np.arange(len(plot_order))
width = 0.25
for offset, metric in enumerate(['test_damage_f1_macro', 'test_damage_f1_micro', 'test_damage_pr_auc_macro']):
    values = benchmark.set_index('model').loc[plot_order, metric].to_numpy()
    axes[0].bar(x + (offset - 1) * width, values, width, label=metric.replace('test_', ''))
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Puntuación')
axes[0].set_title('Comparación global en prueba')
axes[0].set_xticks(x, labels, rotation=25, ha='right')
axes[0].legend()

f1_matrix = np.asarray([
    test_reports[name].loc[COARSE_ORDER, 'f1-score'].to_numpy() for name in plot_order
])
image = axes[1].imshow(f1_matrix, aspect='auto', vmin=0, vmax=1, cmap='YlGnBu')
axes[1].set_title('F1 por categoría gruesa')
axes[1].set_yticks(np.arange(len(plot_order)), labels)
axes[1].set_xticks(np.arange(len(COARSE_ORDER)), [x.replace('_', '\n') for x in COARSE_ORDER], rotation=25, ha='right')
for row in range(f1_matrix.shape[0]):
    for col in range(f1_matrix.shape[1]):
        axes[1].text(col, row, f'{f1_matrix[row, col]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=axes[1], fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'comparacion_modelos_y_f1_categoria.png', dpi=170, bbox_inches='tight')
plt.show()

![Comparación de modelos](../resultados/figuras/moderador_grueso/comparacion_modelos_y_f1_categoria.png)

## 7. ¿El ganador es suficiente para moderar?

No existe un umbral universal de suficiencia. Para este prototipo se declara **antes de leer la prueba** una puerta operativa conservadora: F1 macro calculado solo sobre daños ≥ 0.70 y recall ≥ 0.80 en cada una de las cinco categorías de daño. Además, una afirmación académica final exige evaluación en el holdout humano. Si falla cualquiera, el modelo solo puede priorizar casos para revisión y no decidir sanciones o publicación automáticamente.

In [ ]:
winner_test_row = benchmark.loc[benchmark['model'] == WINNER_NAME].iloc[0]
winner_report = test_reports[WINNER_NAME]
damage_performance = winner_report.loc[DAMAGE_ORDER, ['precision', 'recall', 'f1-score', 'support']].copy()
algorithmic_gate = bool(
    winner_test_row['test_damage_f1_macro'] >= MIN_DAMAGE_MACRO_F1
    and damage_performance['recall'].min() >= MIN_DAMAGE_RECALL
)
human_gate = bool(len(human_ids) > 0)
standalone_sufficient = algorithmic_gate and human_gate

display(damage_performance)
gate_table = pd.DataFrame([
    {'criterio': 'F1 macro de daños', 'resultado': winner_test_row['test_damage_f1_macro'], 'mínimo': MIN_DAMAGE_MACRO_F1, 'cumple': winner_test_row['test_damage_f1_macro'] >= MIN_DAMAGE_MACRO_F1},
    {'criterio': 'Recall mínimo entre daños', 'resultado': damage_performance['recall'].min(), 'mínimo': MIN_DAMAGE_RECALL, 'cumple': damage_performance['recall'].min() >= MIN_DAMAGE_RECALL},
    {'criterio': 'Holdout humano disponible', 'resultado': int(human_gate), 'mínimo': 1, 'cumple': human_gate},
])
display(gate_table)
if standalone_sufficient:
    print('CONCLUSIÓN: supera la puerta provisional; aún debe mantenerse monitoreo humano.')
else:
    print('CONCLUSIÓN: NO es suficiente para moderación autónoma. Solo priorización con revisión humana.')
damage_performance.to_csv(METRICS_DIR / 'desempeno_danos_modelo_ganador.csv')
gate_table.to_csv(METRICS_DIR / 'puerta_suficiencia.csv', index=False)

## 8. Uso de flags transversales para calibrar `needs_review`

Los flags no se predicen como daño. Se usan en validación para elegir el margen de incertidumbre más pequeño que capture al menos 80% de los chunks marcados con ironía, humor encubridor o contexto necesario. Luego esa regla se evalúa una sola vez en prueba. Si alcanzar 80% obliga a revisar casi todo, el resultado también se informa: no se fuerza una conclusión favorable.

In [ ]:
routing_validation = review_routing_curve(
    validation_scores[WINNER_NAME], winner_development.thresholds, validation_df['flags']
)
selected_routing, routing_target_met = select_review_margin(
    routing_validation, minimum_flag_capture=MIN_FLAG_CAPTURE
)
winner_development.review_margin = float(selected_routing['margen'])

routing_test = review_routing_curve(
    test_scores[WINNER_NAME], winner_development.thresholds, test_df['flags'],
    margins=np.asarray([winner_development.review_margin]),
)
display(selected_routing.to_frame('validación'))
display(routing_test)
print('Objetivo de captura de flags alcanzado en validación:', routing_target_met)
routing_validation.to_csv(METRICS_DIR / 'curva_enrutamiento_validacion.csv', index=False)
routing_test.to_csv(METRICS_DIR / 'enrutamiento_prueba.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(routing_validation['tasa_revision'], routing_validation['captura_flags'], marker='o', markersize=3)
ax.scatter(selected_routing['tasa_revision'], selected_routing['captura_flags'], s=100, color='#E45756', label='margen elegido')
ax.axhline(MIN_FLAG_CAPTURE, color='gray', linestyle='--', label='captura objetivo')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Proporción enviada a revisión')
ax.set_ylabel('Proporción de flags capturada')
ax.set_title('Cobertura automática frente a captura de casos transversales')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'curva_revision_flags.png', dpi=170, bbox_inches='tight')
plt.show()

![Curva de revisión y captura de flags](../resultados/figuras/moderador_grueso/curva_revision_flags.png)

## 9. Reajuste del ganador, exportación y validación humana

El ganador se reajusta con entrenamiento + validación, conservando los umbrales y el margen definidos antes de abrir la prueba. El conjunto de prueba y el eventual consenso humano permanecen fuera del ajuste.

In [ ]:
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)
final_model, final_diagnostics = fit_candidate(
    WINNER_NAME, train_validation_df, max_features=MAX_FEATURES
)
final_model.thresholds = winner_development.thresholds.copy()
final_model.review_margin = winner_development.review_margin
final_model.metadata = {
    **hybrid_meta,
    **split_meta,
    'coarse_order': COARSE_ORDER,
    'damage_order': DAMAGE_ORDER,
    'fine_to_coarse': FINE_TO_COARSE,
    'winner_selected_on_validation': WINNER_NAME,
    'selection_metric': 'F1 macro de daños; PR-AUC macro de daños como desempate',
    'review_margin_selected_on_flags': final_model.review_margin,
    'standalone_sufficient_on_pseudo_test': standalone_sufficient,
    'human_validation_available': human_gate,
}
MODEL_PATH = MODEL_DIR / 'moderador_cinco_danos_o_seguro.joblib'
save_coarse_model(final_model, MODEL_PATH)
(MODEL_DIR / 'manifiesto.json').write_text(
    json.dumps(final_model.metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Modelo ganador exportado:', MODEL_PATH)
print('Reajuste final (s):', round(final_diagnostics['training_seconds'], 2))

human_holdout = load_human_holdout(ROOT, FINE_ORDER, FLAG_ORDER)
if human_holdout.empty:
    print('PENDIENTE: no existe consenso humano; no se habilita moderación autónoma.')
else:
    human_holdout = add_coarse_targets(human_holdout, taxonomy)
    fitted_ids = set(train_validation_df['chunk_id']) | set(test_df['chunk_id'])
    assert set(human_holdout['chunk_id']).isdisjoint(fitted_ids)
    human_metrics, human_report, _ = evaluate_candidate(final_model, human_holdout)
    display(pd.Series(human_metrics).to_frame('validación humana'))
    display(human_report)
    human_report.to_csv(METRICS_DIR / 'reporte_validacion_humana.csv')

## 10. Recarga e inferencia

La salida nunca contiene etiquetas finas. `needs_review=True` significa derivar a una persona o modelo de mayor capacidad; no equivale a daño.

In [ ]:
reloaded = load_coarse_model(MODEL_PATH)
examples = [
    'Es una broma, pero las mujeres no sirven para dirigir.',
    'Ese serrano habla horrible y no tiene educación.',
    'Gracias por el análisis; no estoy de acuerdo con la conclusión.',
]
predictions = reloaded.predict(examples)
display(pd.DataFrame([
    {'texto': text, 'etiquetas_gruesas': pred['coarse_labels'], 'needs_review': pred['needs_review']}
    for text, pred in zip(examples, predictions)
]))
del candidate_models
gc.collect()

## Conclusión reproducible

El cuaderno distingue selección y evaluación: el ganador se elige con validación y se juzga una sola vez en prueba agrupada por video. Un valor alto de accuracy no demuestra capacidad de moderación porque `SEGURO` representa cerca del 95% del corpus. La conclusión operativa se deriva de F1 macro y PR‑AUC calculados solo sobre daños, recall por cada daño y disponibilidad de validación humana. Mientras no se superen las puertas declaradas, el artefacto es un **priorizador para revisión humana**, no un moderador autónomo.

El baseline queda congelado en `resultados/INFORME_PRIMER_ENTRENAMIENTO_MODELOS_GRUESOS.md`. Las mejoras posteriores, sin modificar este resultado, se ejecutan en `Cuadernos/04_1_mejoras_entrenamiento_moderador.ipynb` y se documentan en `resultados/INFORME_SEGUNDO_ENTRENAMIENTO_MEJORAS.md`.

### Referencias metodológicas (APA 7)

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

Joachims, T. (1998). Text categorization with support vector machines: Learning with many relevant features. In *Machine Learning: ECML-98* (pp. 137–142). Springer. https://doi.org/10.1007/BFb0026683

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Rennie, J. D. M., Shih, L., Teevan, J., & Karger, D. R. (2003). Tackling the poor assumptions of naive Bayes text classifiers. In *Proceedings of the 20th International Conference on Machine Learning* (pp. 616–623).

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432